# Qwen3.8-27B BF16 — Text + Vision on Kaggle TPU v5e-8

## English

This notebook runs an original BF16 Qwen3.8-27B checkpoint on eight Kaggle TPU v5e devices using one MaxText engine, Split-Vision image encoding, and a JAX/XLA prefill path. It evaluates 19 Text + Vision cases and records prompts, input SHA256, generated text, token IDs, EOS and timing for each case.

**Required attachments:** [Kaggle BF16 model mirror](https://www.kaggle.com/models/dangkhoa2016/qwen-qwen3-8-27b) (Transformers/default/v1) and the matching [Full19 support Dataset](https://www.kaggle.com/datasets/dangkhoa2016/qwen38-27b-tpu-v5e8-full19-runtime-support). The notebook validates the 48-file support manifest before loading checkpoint weights. The Dataset's access and redistribution conditions may limit third-party execution.

This Git file contains executable source cells. Execution creates new output files in `/kaggle/working`; it does not contain pre-filled model responses. The separate recorded TPU measurements in [Full19 evidence](../docs/FULL19-EVIDENCE.md) describe their own pinned inputs and are not an execution claim for this particular file.

## Tiếng Việt

Notebook này thực thi checkpoint Qwen3.8-27B BF16 nguyên gốc trên tám thiết bị Kaggle TPU v5e bằng một MaxText engine, đường mã hóa ảnh Split-Vision và JAX/XLA prefill. Bộ thử gồm 19 tác vụ Text + Vision; mỗi tác vụ lưu prompt, SHA256 ảnh, câu trả lời, token IDs, EOS và thời gian.

**Dữ liệu cần attach:** [Kaggle BF16 model mirror](https://www.kaggle.com/models/dangkhoa2016/qwen-qwen3-8-27b) (Transformers/default/v1) và [Full19 support Dataset](https://www.kaggle.com/datasets/dangkhoa2016/qwen38-27b-tpu-v5e8-full19-runtime-support) tương ứng. Notebook kiểm tra manifest 48 file trước khi nạp trọng số. Quyền truy cập và điều kiện phân phối Dataset có thể hạn chế khả năng chạy lại của người dùng khác.

File Git chứa mã nguồn thực thi; khi chạy, notebook sinh các file kết quả mới trong `/kaggle/working` và không nhúng sẵn câu trả lời mô hình. Các số đo TPU trong [tài liệu bằng chứng Full19](../docs/FULL19-EVIDENCE.vi.md) gắn với bộ input của lượt đo đó, không mặc nhiên là kết quả chạy file này.

## Execution and evidence / Thực thi và bằng chứng

- Verify the model attachment, 48 support-file hashes, dependency versions and eight-device TPU topology before loading weights.
- Run all cells sequentially in one TPU session with one BF16 engine. Do not restart the kernel between model initialization and inference.
- Generate a unique UTC-stamped evidence directory. Store raw outputs, token IDs, EOS flags, timings and exceptions for all cases; do not replace the generated content with earlier examples.
- Report successful generation separately from factual accuracy. Numerical OCR and image reasoning should also be reviewed against the input images.
- Reused JAX cache files still require native sharding, dtype, shape and layout validation.

**Tiếng Việt:** Xác minh model, SHA256 của 48 file hỗ trợ, phiên bản thư viện và tám thiết bị TPU trước khi nạp trọng số. Chạy lần lượt các cell trong một phiên TPU; mỗi lần chạy ghi bằng chứng riêng theo UTC, bao gồm output nguyên văn, token IDs, EOS và thời gian. Đánh giá độ chính xác nội dung độc lập với việc sinh token thành công; cache JAX vẫn phải qua kiểm tra tương thích native.

In [ ]:
# 1. Fail-closed, offline and portable preflight: run before importing JAX.
import os, json, hashlib, time
from datetime import datetime, timezone
from pathlib import Path
RUN_WALL_START=time.monotonic()
RUN_STARTED_UTC=datetime.now(timezone.utc).isoformat()
MODEL_DIR=Path(os.environ.get('QWEN38_MODEL_DIR',
    '/kaggle/input/models/dangkhoa2016/qwen-qwen3-8-27b/transformers/default/1'))
assert MODEL_DIR.is_dir() and (MODEL_DIR/'model.safetensors.index.json').is_file(),('MODEL_MOUNT_MISSING',str(MODEL_DIR))

# Kaggle Dataset upload expands the verified ZIP into a filename-derived subdirectory.
# Accept both its observed mount structure and a manually extracted flat support folder.
PORTABLE_SUPPORT_SUBDIR='.'
SUPPORT_CANDIDATES=[
    Path(os.environ['QWEN38_SUPPORT_DIR']) if os.environ.get('QWEN38_SUPPORT_DIR') else None,
    Path('/kaggle/input/datasets/dangkhoa2016/qwen38-27b-tpu-v5e8-full19-runtime-support'),
    Path('/kaggle/input/qwen38-27b-tpu-v5e8-full19-runtime-support'),
]
def find_portable_support_folder(candidates, subdirectory):
    for base in candidates:
        if base is None:
            continue
        for candidate in (base,base/subdirectory):
            if (candidate/'portable-bundle-manifest.json').is_file():
                return candidate
    return None
SUPPORT_DIR=find_portable_support_folder(SUPPORT_CANDIDATES,PORTABLE_SUPPORT_SUBDIR)
assert SUPPORT_DIR is not None,('OFFLINE_SUPPORT_NOT_ATTACHED',
    [str(x) for x in SUPPORT_CANDIDATES if x is not None],PORTABLE_SUPPORT_SUBDIR)
BUNDLE_MANIFEST=SUPPORT_DIR/'portable-bundle-manifest.json'
PORTABLE_MANIFEST_SHA256='cf5343f614eb32ff9234c870e4a26093f9f07e62199ee90396d516a804eb8a28'
def file_sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
assert file_sha256(BUNDLE_MANIFEST)==PORTABLE_MANIFEST_SHA256,'PORTABLE_MANIFEST_SHA256_MISMATCH'
bundle=json.loads(BUNDLE_MANIFEST.read_text())
assert bundle['format']=='QWEN38_FULL19_PORTABLE_SUPPORT_V1'
assert bundle['required_file_count']==48 and len(bundle['files'])==48
assert bundle['support_file_count']==38 and bundle['source_input_images']==16
for relative,item in bundle['files'].items():
    rp=Path(relative)
    assert not rp.is_absolute() and '..' not in rp.parts
    file=SUPPORT_DIR/rp
    assert file.is_file() and file.stat().st_size==item['bytes'],('PORTABLE_FILE_MISSING',relative)
    assert file_sha256(file)==item['sha256'],('PORTABLE_FILE_SHA256_MISMATCH',relative)
assert (SUPPORT_DIR/'split_vision_maxtext.patch').is_file()
assert (SUPPORT_DIR/'pinned-cache-manifest.json').is_file()
print('PORTABLE_SUPPORT_BUNDLE_SHA256_GATE_PASS',len(bundle['files']),str(SUPPORT_DIR))

os.environ.setdefault('QWEN38_ALLOW_PINNED_WEB_WHEELS','0')
os.environ.update({
 'PJRT_DEVICE':'TPU','TPU_SKIP_MDS_QUERY':'1',
 'TPU_HOST_BOUNDS':'1,1,1','TPU_CHIPS_PER_HOST_BOUNDS':'2,4,1',
 'TPU_WORKER_HOSTNAMES':os.uname().nodename,'TPU_WORKER_ID':'0',
 'TPU_ACCELERATOR_TYPE':'v5e-8','JAX_PLATFORMS':'tpu,cpu',
})
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_ROOT=Path('/kaggle/working')/('qwen38-full19-production-'+RUN_ID)
assert not RUN_ROOT.exists()
RUN_ROOT.mkdir(exist_ok=False)
MAXTEXT_DIR=RUN_ROOT/'patched-maxtext'
print('RUN_ROOT',RUN_ROOT,'MODEL_DIR',MODEL_DIR)


In [ ]:
# 2. Offline-first dependency gate — never replace Kaggle's JAX/TPU stack.
import hashlib
import importlib.metadata as md
import subprocess, sys, json
from pathlib import Path

# The portable support manifest and all 48 payload files were verified in Cell 1.
assert SUPPORT_DIR.is_dir() and BUNDLE_MANIFEST.is_file(), ('SUPPORT_DATASET_MISSING',str(SUPPORT_DIR))
assert len(bundle['files']) == 48
print('SUPPORT_MANIFEST_VALIDATED',PORTABLE_MANIFEST_SHA256)

WHEEL_INFO = {
    'pathwaysutils': ('0.1.12', 'pathwaysutils-0.1.12-py3-none-any.whl', '3aab3cd936ea9076c469b731cbdca43a453736594c65bdcee4b274c10ce51c34'),
    'aqtp': ('0.9.0', 'aqtp-0.9.0-py3-none-any.whl', '5efdccd24657e149d3ac17599bb3d4eb88297fdc6aab14eaba0c70625057a257'),
    'drjax': ('0.2.1', 'drjax-0.2.1-py3-none-any.whl', '9ec93541a7ba0af864d599cd3e14770a19e291a6291b8b79ddf6b92e8b173494'),
    'tokamax': ('0.0.14', 'tokamax-0.0.14-py3-none-any.whl', 'a10a91bbb6ddf406bb7006cb1a27af59a6faaa6cb06fe788249dd10d9eaac409'),
    'xprof': ('2.23.1', 'xprof-2.23.1-py3-none-manylinux_2_27_x86_64.whl', '96701f868b7a736a789d0a4b5efeeb269feaed63348117707e853a72e006c722'),
    'einshape': ('1.0', 'einshape-1.0-py3-none-any.whl', '42da4c2dea3a27f87ee45a7cee5072a636b97cb184bb07bf5d6412ba0ff7b965'),
    'typeguard': ('2.13.3', 'typeguard-2.13.3-py3-none-any.whl', '5e3e3be01e887e7eafae5af63d1f36c849aaa94e3a0112097312aabfa16284f1'),
    'tiktoken': ('0.14.0', 'tiktoken-0.14.0-cp312-cp312-manylinux_2_28_x86_64.whl', '7896eea257fe497a2b7134474d909156c6744ce8da35bce88011a960e008aa0d'),
}

# Additional version-pinned wheels used by the MaxText runtime.
extra_manifest=json.loads((SUPPORT_DIR/'manifest.json').read_text())['wheels']
assert {x['dist'] for x in extra_manifest}=={'jaxtyping','wadler-lindig','qwix'}
for x in extra_manifest:
    WHEEL_INFO[x['dist']]=(x['version'],x['wheel'],x['sha256'])

CORE_EXPECTED = {'jax':'0.11.0','jaxlib':'0.11.0','flax':'0.12.8','transformers':'5.12.1'}
core_versions = {k:md.version(k) for k in CORE_EXPECTED}
assert core_versions==CORE_EXPECTED,('BASELINE_RUNTIME_MISMATCH',core_versions,CORE_EXPECTED)
print('CORE_BASELINE_PASS',core_versions)

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

for dist, (ver, wheel_name, expected_sha) in WHEEL_INFO.items():
    try: current=md.version(dist)
    except md.PackageNotFoundError: current=None
    if current != ver:
        wheel = SUPPORT_DIR / 'wheels' / wheel_name
        if not wheel.is_file():
            wheel = SUPPORT_DIR / wheel_name
        assert wheel.is_file(), ('PINNED_WHEEL_NOT_ATTACHED',dist,wheel_name)
        assert sha256(wheel) == expected_sha, ('WHEEL_SHA_MISMATCH',wheel_name)
        subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-index','--no-deps',str(wheel)])
    assert md.version(dist) == ver, (dist, md.version(dist), ver)

# MaxText's pathwaysutils profiling module imports FastAPI transitively.
# Pin the validated compatibility set without letting pip replace JAX/JAXLIB.
WEB_IMPORT_DEPS = {'fastapi':'0.115.12', 'starlette':'0.46.2', 'uvicorn':'0.34.2'}
allow_online=os.environ.get('QWEN38_ALLOW_PINNED_WEB_WHEELS','0')=='1'
for dist,ver in WEB_IMPORT_DEPS.items():
    try: current=md.version(dist)
    except md.PackageNotFoundError: current=None
    if current != ver:
        candidates=sorted(SUPPORT_DIR.glob(f'{dist.replace("-", "_")}-{ver}-*.whl'))
        if candidates:
            wheel=candidates[0]
            # Non-mutating integrity check above already pinned this file via full-dataset SHA256.
            subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-index','--no-deps',str(wheel)])
        else:
            assert allow_online, ('WEB_IMPORT_WHEELS_MISSING',dist,ver,
                  'Attach the documented Full19 support Dataset, or enable QWEN38_ALLOW_PINNED_WEB_WHEELS=1 with Kaggle Internet.')
            subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps',f'{dist}=={ver}'])
    assert md.version(dist)==ver
print('MAXTEXT_TRANSITIVE_IMPORT_GATE=PASS', WEB_IMPORT_DEPS)

CORE_EXPECTED = {'jax': '0.11.0', 'jaxlib': '0.11.0', 'transformers': '5.12.1'}
versions = {k: md.version(k) for k in CORE_EXPECTED}
print('Core versions:', versions)
mismatches={k:(versions[k],v) for k,v in CORE_EXPECTED.items() if versions[k] != v}
assert not mismatches, f'Fail-closed: core runtime differs from acceptance environment: {mismatches}. Do not pip-replace Kaggle JAX/JAXLIB in a live TPU runtime.'
print('DEPENDENCY_GATE=PASS', 'offline_support='+str(SUPPORT_DIR.is_dir()))

In [ ]:
# 3. Offline MaxText bootstrap with exact source/patch provenance.
import hashlib, json, shutil, tarfile
from pathlib import Path

MAXTEXT_COMMIT = 'f0b2abe3f6f3736c5579673f42b5cbc17b327fd3'
MAXTEXT_DIR=RUN_ROOT/'patched-maxtext'
PATCH_SHA256 = '07f27724c7e6d7267258eab99bfbe3172329ca359959a36f73fee29531acb8e6'
OFFLINE_ARCHIVE_SHA256 = '1878a1fcbd8d4bc0e3747b2ce17bd6f40616035ca88b6c177022a3b2de0a7e13'

def file_sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

assert SUPPORT_DIR.is_dir(), f'Offline support dataset not attached: {SUPPORT_DIR}'
PATCH_PATH = SUPPORT_DIR / 'maxtext-qwen38-production-text-vision.patch'
assert PATCH_PATH.is_file(), f'Missing MaxText patch provenance file: {PATCH_PATH}'
assert file_sha256(PATCH_PATH) == PATCH_SHA256, 'MaxText patch SHA mismatch'

archive_name = 'maxtext-f0b2abe-qwen38-text-vision-patched.tar.gz'
archive_candidates = [
    SUPPORT_DIR / 'src' / archive_name,
    SUPPORT_DIR / archive_name,
    SUPPORT_DIR / (archive_name + '.blob'),
]
archive = next((p for p in archive_candidates if p.is_file()), archive_candidates[0])
assert archive.is_file(), f'Missing patched MaxText archive: {archive_candidates}'
assert file_sha256(archive) == OFFLINE_ARCHIVE_SHA256, 'Offline MaxText archive SHA mismatch'

assert not MAXTEXT_DIR.exists(), ('NON_DESTRUCTIVE_BOOTSTRAP_TARGET_EXISTS',str(MAXTEXT_DIR))
MAXTEXT_DIR.mkdir(parents=True,exist_ok=False)
with tarfile.open(archive,'r:gz') as tf:
    tf.extractall(MAXTEXT_DIR,filter='data')
meta=json.loads((MAXTEXT_DIR/'QWEN38_OFFLINE_BUILD.json').read_text())
assert meta['maxtext_commit'] == MAXTEXT_COMMIT
# The embedded build marker records the original pre-cast patch. The archive and
# separate integrated patch are pinned above; verify the actual loader bytes too.
assert meta['patch_sha256']=='2ec8241c93b3a3e1d54e97e85eace644d27f90ba2147c1bc8059bf67998c38cf'
loader_path=MAXTEXT_DIR/'src/maxtext/checkpoint_conversion/utils/tensor_handling.py'
assert file_sha256(loader_path)=='4f663cfae0d7d89e9c2c72ac5331dfb73e8e83e591e2261015cb316e8b8b5218'
assert 'array.astype(leaf.dtype)' in loader_path.read_text()

# The SHA256-verified source archive provides the pinned MaxText provenance.
assert 'Q/K RMSNorm' in meta.get('qk_norm_fix','')
source_mode='offline-support-dataset'

marker = MAXTEXT_DIR / 'src/maxtext/configs/models/qwen3.8-27b.yml'
assert marker.exists(), 'Qwen3.8 MaxText patch marker missing'
print('MaxText commit:', MAXTEXT_COMMIT)
print('Patch SHA256:', PATCH_SHA256)
print('Source mode:', source_mode)
print('Qwen3.8 patch: READY')

# Reproduce the original SHA-verified Split-Vision patch; keep source integrity gates.
import subprocess
SPLIT_PATCH=SUPPORT_DIR/'split_vision_maxtext.patch'
assert SPLIT_PATCH.is_file(), 'Split Vision patch missing; extract local integration kit first'
assert file_sha256(SPLIT_PATCH)=='7ba8d1bdedbfab2c8338c101ca94f51580caf4d4ee80982225f58fe5daec5a65', 'Split patch SHA mismatch'
for rel,original_sha,patched_sha in [('maxtext/models/models.py', '13fe44df690cd5a21dfaee2aeba2b675c3d4ff888056345cc81237f936927e52', 'bd2cdead53ded2f8bafa9eb5f0500c622d9348cd7f41a81b2c16377f712e8e95'), ('maxtext/inference/maxengine/maxengine.py', 'ea6e5b143f875be451bc63b3e7474767e592e9766dc5f218e0109ef95d399b07', '6ee4040370cb76e4227fe6b7cbc492038809d6940e95aab4008d3365e55f192e')]:
    p=MAXTEXT_DIR/'src'/rel
    assert file_sha256(p)==original_sha, ('SPLIT_PATCH_SOURCE_BASE_MISMATCH',rel)
subprocess.run(['git','apply','--check',str(SPLIT_PATCH)],cwd=MAXTEXT_DIR/'src',check=True)
subprocess.run(['git','apply',str(SPLIT_PATCH)],cwd=MAXTEXT_DIR/'src',check=True)
for rel,original_sha,patched_sha in [('maxtext/models/models.py', '13fe44df690cd5a21dfaee2aeba2b675c3d4ff888056345cc81237f936927e52', 'bd2cdead53ded2f8bafa9eb5f0500c622d9348cd7f41a81b2c16377f712e8e95'), ('maxtext/inference/maxengine/maxengine.py', 'ea6e5b143f875be451bc63b3e7474767e592e9766dc5f218e0109ef95d399b07', '6ee4040370cb76e4227fe6b7cbc492038809d6940e95aab4008d3365e55f192e')]:
    assert file_sha256(MAXTEXT_DIR/'src'/rel)==patched_sha, ('SPLIT_PATCH_INTEGRITY_FAIL',rel)
print('SPLIT_VISION_SOURCE_GATE=PASS',len([{'file': 'maxtext/models/models.py', 'original_sha256': '13fe44df690cd5a21dfaee2aeba2b675c3d4ff888056345cc81237f936927e52', 'patched_sha256': 'bd2cdead53ded2f8bafa9eb5f0500c622d9348cd7f41a81b2c16377f712e8e95', 'added_lines': 15, 'removed_lines': 1}, {'file': 'maxtext/inference/maxengine/maxengine.py', 'original_sha256': 'ea6e5b143f875be451bc63b3e7474767e592e9766dc5f218e0109ef95d399b07', 'patched_sha256': '6ee4040370cb76e4227fe6b7cbc492038809d6940e95aab4008d3365e55f192e', 'added_lines': 32, 'removed_lines': 0}]))


In [ ]:
# 3b. Verify image assets and case definitions from the attached support Dataset.
EXAMPLES_DIR=SUPPORT_DIR  # all 16 inputs are in this attached verified bundle
assets=json.loads((SUPPORT_DIR/'assets_manifest.json').read_text())['assets']
assert len(assets)==16,len(assets)
for item in assets:
    p=SUPPORT_DIR/Path(item['path']).name
    assert p.is_file() and hashlib.sha256(p.read_bytes()).hexdigest()==item['sha256'],('ASSET_SHA_MISMATCH',p)
rich_spec=json.loads((SUPPORT_DIR/'vision_showcase_cases.json').read_text())['cases']
photo_spec=json.loads((SUPPORT_DIR/'photo_showcase_cases.json').read_text())['cases']
assert len(rich_spec)==6 and len(photo_spec)==8
print('SINGLE_DATASET_ASSET_GATE_PASS',len(assets),len(rich_spec),len(photo_spec))


In [ ]:
# 4. Source checkpoint audit before TPU-heavy work.
import json
from pathlib import Path

cfg = json.loads((MODEL_DIR / 'config.json').read_text())
index = json.loads((MODEL_DIR / 'model.safetensors.index.json').read_text())
shards = sorted(MODEL_DIR.glob('model-*.safetensors'))
print('architecture:', cfg.get('architectures'))
print('model_type:', cfg.get('model_type'))
print('safetensors shards:', len(shards))
print('declared total bytes:', index.get('metadata', {}).get('total_size'))
assert len(shards) == 18, f'Expected 18 shards, found {len(shards)}'
assert index.get('metadata', {}).get('total_size') == 55_562_855_904
print('CHECKPOINT_AUDIT=PASS')

In [ ]:
# 5. Initialize JAX TPU and patched MaxText.
import sys, time
os.environ['MAXTEXT_REPO_ROOT'] = str(MAXTEXT_DIR)
sys.path.insert(0, str(MAXTEXT_DIR / 'src'))

import jax
import jax.numpy as jnp
from transformers import AutoTokenizer
from maxtext.configs import pyconfig
from maxtext.inference.maxengine.maxengine import MaxEngine
from maxtext.utils.globals import MAXTEXT_REPO_ROOT

print('backend=', jax.default_backend())
print('device_count=', jax.device_count())
print('local_device_count=', jax.local_device_count())
assert jax.default_backend() == 'tpu'
assert jax.device_count() == 8
print('TPU_GATE=PASS')

In [ ]:
# 6. One multimodal engine, one BF16 load.
from maxtext.multimodal import processor as mm_processor
from maxtext.multimodal import processor_qwen3_omni as qwen_mm

import importlib.util
spec=importlib.util.spec_from_file_location('qwen38_engine_args',SUPPORT_DIR/'engine_args.py')
e_args=importlib.util.module_from_spec(spec);spec.loader.exec_module(e_args)
# Verified shape: 320-token prefill, 768-token target. Rich/Photography decode up to 448 tokens.
e_args.MAX_TARGET_LEN=768
e_args.PREFILL_LEN=320
e_args.INFERENCE_WIDTH=512
e_args.INFERENCE_HEIGHT=384
PREFILL_LEN=e_args.PREFILL_LEN
MAX_TARGET_LEN=e_args.MAX_TARGET_LEN
print('COMPILE_TARGET_LENGTH',MAX_TARGET_LEN,'PREFILL',PREFILL_LEN)
VISION_SEED_IMAGE=EXAMPLES_DIR/'vision_test_card_731_1280x960.png'
assert VISION_SEED_IMAGE.is_file()
args=e_args.build_engine_args(MAXTEXT_REPO_ROOT,MODEL_DIR,VISION_SEED_IMAGE)
# Verify the optional compilation cache without modifying the user cache.
STAGE_MANIFEST=SUPPORT_DIR/'pinned-cache-manifest.json'
STAGE_METADATA=json.loads(STAGE_MANIFEST.read_text())
assert len(STAGE_METADATA['entries'])==3
assert len(STAGE_METADATA['entries'])==3
STAGE_CACHE=RUN_ROOT/'jax_cache'
assert not STAGE_CACHE.exists()
STAGE_CACHE.mkdir()
import shutil
for entry in STAGE_METADATA['entries']:
    source=SUPPORT_DIR/entry['file']
    assert source.is_file() and file_sha256(source)==entry['sha256']
    target=STAGE_CACHE/entry['file']
    shutil.copyfile(source,target)
    assert target.stat().st_size==entry['bytes'] and file_sha256(target)==entry['sha256']
assert not any(x.startswith('jax_cache_dir=') for x in args)
args.append('jax_cache_dir='+str(STAGE_CACHE))
print('JAX_CACHE_CANDIDATE_SHA256_PASS',len(STAGE_METADATA['entries']))
print('NATIVE_JAX_SHARDING_LAYOUT_CHECKS_REQUIRED; CACHE_HIT_NOT_GUARANTEED')

t0=time.time(); config=pyconfig.initialize(args); engine=MaxEngine(config); construct_s=time.time()-t0
print(f'engine_construct_seconds={construct_s:.3f}')
t0=time.time(); params=engine.load_params(); load_s=time.time()-t0
print(f'load_params_seconds={load_s:.3f}')
assert getattr(config,'use_multimodal',False) is True
print('MODEL_LOAD=PASS')
print('ONE_ENGINE_POLICY=PASS')

In [ ]:
# 7. Shared deterministic helpers.
import numpy as np, statistics
from IPython.display import display, Image as IPImage

tok = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True, trust_remote_code=False)
pad_id = tok.pad_token_id if tok.pad_token_id is not None else (tok.eos_token_id or 0)

from PIL import Image as PILImage
import hashlib
RESIZED_DIR=RUN_ROOT/'resized-inputs-512x384'
RESIZED_DIR.mkdir(exist_ok=True,parents=True)
def image_for_inference(path):
    path=Path(path)
    assert path.is_file(),path
    dest=RESIZED_DIR/(path.stem+'-'+hashlib.sha256(path.read_bytes()).hexdigest()[:12]+'.png')
    if not dest.is_file():
        with PILImage.open(path) as im:
            converted=im.convert('RGB').resize((e_args.INFERENCE_WIDTH,e_args.INFERENCE_HEIGHT),PILImage.Resampling.LANCZOS)
            converted.save(dest)
    with PILImage.open(dest) as im:
        assert im.size==(e_args.INFERENCE_WIDTH,e_args.INFERENCE_HEIGHT)
    return dest

def completion_info(ids):
    eos=int(tok.eos_token_id)
    return {'eos_reached':bool(ids and int(ids[-1])==eos),'completion_status':'EOS' if ids and int(ids[-1])==eos else 'MAX_STEPS'}

TEXT_SUFFIX=' Answer briefly in one complete sentence.'
VISION_SUFFIX=' Be concise, factual, and avoid speculation.'

def _decode(prefix, first, true_length, max_decode_steps):
    state=engine.init_decode_state(); state=engine.insert(prefix,state,slot=0)
    out=[int(jax.device_get(first.data)[0,0])]; step_s=[]
    if out[0] == tok.eos_token_id: return out, tok.decode(out, skip_special_tokens=False), step_s
    for _ in range(max_decode_steps-1):
        t=time.time(); state,res=engine.generate(params=params,decode_state=state,algorithm='greedy'); jax.block_until_ready(res.data); step_s.append(time.time()-t)
        tid=int(jax.device_get(res.data)[0,0]); out.append(tid)
        if tid == tok.eos_token_id: break
    return out, tok.decode(out, skip_special_tokens=False), step_s

def generate_text(prompt: str, max_decode_steps: int=96):
    rendered=tok.apply_chat_template([{'role':'user','content':prompt+TEXT_SUFFIX}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    ids=tok(rendered, add_special_tokens=False)['input_ids']
    assert len(ids) <= PREFILL_LEN, (len(ids), PREFILL_LEN, prompt)
    padded=np.asarray(ids+[pad_id]*(PREFILL_LEN-len(ids)),dtype=np.int32)
    t=time.time(); prefix,first=engine.prefill(params=params,padded_tokens=jnp.asarray(padded),true_length=len(ids),algorithm='greedy'); jax.block_until_ready(first.data); pre=time.time()-t
    out,text,steps=_decode(prefix,first,len(ids),max_decode_steps)
    return {'prompt':prompt,'prompt_token_count':len(ids),**completion_info(out),'generated_token_ids':out,'generated_text':text,'prefill_seconds':pre,'decode_step_seconds':steps}

def generate_vision(image_path, prompt: str, max_decode_steps: int=224):
    image_path=Path(image_path); assert image_path.is_file(), image_path
    resized=image_for_inference(image_path)
    rendered_prompt=prompt+VISION_SUFFIX
    pd=config._pydantic_config.model_copy(update={'image_path':str(resized),'prompt':rendered_prompt})
    cfg=pyconfig.HyperParameters(pd)
    po=mm_processor.preprocess_mm_data(cfg)
    off=int(mm_processor.get_image_offsets(config=cfg,processor_output=po))
    content=f'<|vision_start|><|image_pad|><|vision_end|>{rendered_prompt}'
    rendered=tok.apply_chat_template([{'role':'user','content':content}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    base=tok(rendered,add_special_tokens=False)['input_ids']
    text_budget=PREFILL_LEN-off
    assert len(base) <= text_budget, {'text_tokens':len(base),'image_offset':off,'prefill':PREFILL_LEN}
    padded=np.asarray(base+[pad_id]*(text_budget-len(base)),dtype=np.int32)
    fused=mm_processor.prepare_text_for_image_fusion(tokens=padded,config=cfg,processor_output=po)
    true_length=len(base)+off
    mask=(np.arange(len(fused))[None,:] < true_length).astype(np.int32)
    pos,delta=qwen_mm.get_rope_index(
        input_ids=fused[None,:], image_grid_thw=po.pixel_grid_thw, attention_mask=mask,
        spatial_merge_size=cfg.spatial_merge_size_for_vit,
        position_id_per_seconds=cfg.position_id_per_seconds, config=cfg,
    )
    t=time.time(); prefix,first=engine.prefill(
        params=params,padded_tokens=jnp.asarray(fused,dtype=jnp.int32),positions=jnp.asarray(pos),mrope_deltas=jnp.asarray(delta),
        images=jnp.asarray(po.pixel_values),true_length=true_length,algorithm='greedy'); jax.block_until_ready(first.data); pre=time.time()-t
    max_decode_steps=min(max_decode_steps, MAX_TARGET_LEN-true_length)
    out,text,steps=_decode(prefix,first,true_length,max_decode_steps)
    return {'image':str(image_path),'prompt':prompt,**completion_info(out),'grid':po.pixel_grid_thw.tolist(),'image_offset_tokens':off,'fused_true_length':true_length,'generated_token_ids':out,'generated_text':text,'prefill_seconds':pre,'decode_step_seconds':steps}

print('SHARED_GENERATION_HELPERS=READY')

In [ ]:
# The three live Text cases run at the end on the same BF16 engine.
print('TEXT_CASES_SCHEDULED_FOR_LIVE_FULL19')


In [ ]:
# One real OCR inference with phase milestones saved OUTSIDE the notebook.
# No hardcoded model response. Sidecar survives a SIGALRM during TPU compilation.
import json, time
from datetime import datetime,timezone
from pathlib import Path
import numpy as np
TRACE_PATH=RUN_ROOT/'ocr731-phase-events.json'
TRACE_PATH.parent.mkdir(parents=True,exist_ok=True)
assert not TRACE_PATH.exists(), 'Evidence already exists: do not overwrite or rerun.'
TRACE={'status':'RUNNING','started_utc':datetime.now(timezone.utc).isoformat(),'events':[],'target':'real OCR 731, TPU v5e-8'}
STEP_START=time.monotonic()
def mark(event,**metadata):
    TRACE['events'].append({'phase':event,'elapsed_s':round(time.monotonic()-STEP_START,3),
        'utc':datetime.now(timezone.utc).isoformat(),**metadata})
    tmp=TRACE_PATH.with_suffix('.writing')
    tmp.write_text(json.dumps(TRACE,indent=2,ensure_ascii=False)+'\n')
    tmp.replace(TRACE_PATH)
    print('OCR_PHASE',event,round(time.monotonic()-STEP_START,3),metadata,flush=True)
mark('BEGIN',jax_cache_dir=str(STAGE_CACHE),manifest_sha256=PORTABLE_MANIFEST_SHA256)
ocr_source=EXAMPLES_DIR/'vision_test_card_731_1280x960.png'
assert ocr_source.is_file()
resized=image_for_inference(ocr_source)
mark('IMAGE_RESIZED',image=str(resized))
rendered_prompt='Read the large number in this image. Answer only the digits.'+VISION_SUFFIX
pd=config._pydantic_config.model_copy(update={'image_path':str(resized),'prompt':rendered_prompt})
cfg=pyconfig.HyperParameters(pd)
po=mm_processor.preprocess_mm_data(cfg)
off=int(mm_processor.get_image_offsets(config=cfg,processor_output=po))
content=f'<|vision_start|><|image_pad|><|vision_end|>{rendered_prompt}'
rendered=tok.apply_chat_template([{'role':'user','content':content}],tokenize=False,add_generation_prompt=True,enable_thinking=False)
base=tok(rendered,add_special_tokens=False)['input_ids']
budget=PREFILL_LEN-off
assert len(base)<=budget,(len(base),budget)
padded=np.asarray(base+[pad_id]*(budget-len(base)),dtype=np.int32)
fused=mm_processor.prepare_text_for_image_fusion(tokens=padded,config=cfg,processor_output=po)
true_length=len(base)+off
mask=(np.arange(len(fused))[None,:]<true_length).astype(np.int32)
pos,delta=qwen_mm.get_rope_index(input_ids=fused[None,:],image_grid_thw=po.pixel_grid_thw,
    attention_mask=mask,spatial_merge_size=cfg.spatial_merge_size_for_vit,
    position_id_per_seconds=cfg.position_id_per_seconds,config=cfg)
mark('PREPROCESS_READY',image_offset_tokens=off,prefill_tokens=true_length,
    grid=po.pixel_grid_thw.tolist(),pixel_shape=list(po.pixel_values.shape))
# Compile/run Vision encoder separately. The exact result is forwarded to the decoder without modification.
t=time.monotonic()
mark('VISION_ENCODE_ENTER')
image_embeddings,deepstack_features=engine.encode_vision(
    params=params,images=jnp.asarray(po.pixel_values))
mark('VISION_ENCODE_DISPATCH_RETURNED')
jax.block_until_ready(image_embeddings)
for feature in (deepstack_features or []):
    jax.block_until_ready(feature)
vision_s=time.monotonic()-t
mark('VISION_ENCODE_SYNC_COMPLETE',vision_seconds=round(vision_s,3),
    embedding_shape=list(image_embeddings.shape),
    deepstack_count=len(deepstack_features or []))
# JAX AOT diagnostic: individually instrument lowering, compile/loading, executable dispatch and sync.
# Match the previous engine.prefill call exactly, but call the compiled object directly.
t=time.monotonic()
if engine.rng is None:
    engine.rng=jax.random.PRNGKey(0)
engine.rng,aot_rng=jax.random.split(engine.rng)
aot_kwargs=dict(
    params=params,existing_prefix=None,
    padded_tokens=jnp.asarray(fused,dtype=jnp.int32),
    positions=jnp.asarray(pos),mrope_deltas=jnp.asarray(delta),
    images=None,image_embeddings=image_embeddings,deepstack_visual_embeds=deepstack_features,
    image_masks=None,videos=None,video_masks=None,video_grid_thw=None,
    audio_values=None,audio_masks=None,true_length=true_length,sampler=None,
    rng=aot_rng,slot=None,return_prompt_logp=False,algorithm='greedy',
    topk=None,nucleus_topp=None,temperature=None)
static_keys=('video_grid_thw','return_prompt_logp','algorithm','topk','nucleus_topp')
dynamic_kwargs={k:v for k,v in aot_kwargs.items() if k not in static_keys}
mark('DECODER_AOT_LOWER_ENTER')
lowered=engine._prefill_jit.lower(engine,**aot_kwargs)
mark('DECODER_AOT_LOWER_RETURN')
# Verified on previous experimental TPU run; native correctness checks are retained.
import sys
sys.path.insert(0,str(SUPPORT_DIR))
from aot_jax_pxla_phase_hooks import install as install_aot_phase_hooks
from aot_pxla_nested_sharding_hooks import install as install_nested_sharding_hooks
from aot_native_metadata_hooks import install as install_native_metadata_hooks
from aot_sharding_metadata_reuse_strict import install as install_metadata_reuse
LAYOUT_HOOK_FILE=SUPPORT_DIR/'aot_native_layout_timing_hooks.py'
assert LAYOUT_HOOK_FILE.is_file() and hashlib.sha256(LAYOUT_HOOK_FILE.read_bytes()).hexdigest()=='ac58045e8ede38f5434a9761abc2001472d68b9012451c847e1d6d841325bce8', 'LAYOUT_HOOK_HELPER_SHA_MISMATCH'
sys.path.insert(0,str(LAYOUT_HOOK_FILE.parent))
from aot_native_layout_timing_hooks import install as install_native_layout_timing
from contextlib import ExitStack
mark('DECODER_PXLA_HOOK_INSTALL_ENTER')
with ExitStack() as cleanups:
    remove_aot_hooks=install_aot_phase_hooks(lambda label:mark(label))
    cleanups.callback(remove_aot_hooks)
    remove_native_hooks=install_native_metadata_hooks(lambda label:mark(label))
    cleanups.callback(remove_native_hooks)
    remove_reuse_hooks=install_metadata_reuse(lambda label:mark(label))
    def restore_memo():
        stats=remove_reuse_hooks()
        mark('DECODER_METADATA_REUSE_STATS',**stats)
    cleanups.callback(restore_memo)
    remove_nested_hooks=install_nested_sharding_hooks(lambda label:mark(label))
    cleanups.callback(remove_nested_hooks)
    remove_native_layout_timing=install_native_layout_timing(lambda label:mark(label))
    cleanups.callback(remove_native_layout_timing)
    mark('DECODER_PXLA_HOOK_INSTALL_RETURN')
    mark('DECODER_AOT_COMPILE_ENTER')
    compiled=lowered.compile()
    mark('DECODER_AOT_COMPILE_RETURN')
mark('DECODER_PXLA_HOOK_REMOVED')
mark('DECODER_EXEC_ENTER')
prefix,first=compiled(**dynamic_kwargs)
mark('DECODER_EXEC_RETURN')
jax.block_until_ready(first.data)
prefill_s=time.monotonic()-t
mark('PREFILL_RETURN',decoder_aot_total_seconds=round(prefill_s,3))
t=time.monotonic()
mark('DECODE_ENTER')
out,text,steps=_decode(prefix,first,true_length,max_decode_steps=12)
mark('DECODE_RETURN',decode_seconds=round(time.monotonic()-t,3),decode_tokens=len(out))
ocr_result={'prompt':rendered_prompt,'image':str(ocr_source),'generated_text':text,
    'generated_token_ids':out,'eos_reached':completion_info(out)['eos_reached'],
    'semantic_check':'731' in text,'vision_seconds':vision_s,'prefill_seconds':prefill_s,
    'decode_step_seconds':steps,'image_offset_tokens':off,'true_length':true_length}
TRACE['status']='COMPLETE'
TRACE['ocr_result']=ocr_result
mark('DONE')
print('LIVE_OCR',json.dumps({'generated_text':text,'semantic_check':'731' in text,
    'eos_reached':ocr_result['eos_reached'],'vision_seconds':vision_s,'prefill_seconds':prefill_s,
    'decode_tokens':len(out)},ensure_ascii=False),flush=True)


In [ ]:
# Final evidence is separate from the incremental sidecar.
from datetime import datetime, timezone
final=RUN_ROOT/'ocr731-bootstrap-summary.json'
final.parent.mkdir(parents=True,exist_ok=True)
assert not final.exists()
final.write_text(json.dumps({'status':'COMPLETED_NEEDS_REVIEW',
 'utc':datetime.now(timezone.utc).isoformat(),
 'construct_seconds':construct_s,'checkpoint_load_seconds':load_s,
 'ocr':ocr_result,'full19_executed':False},indent=2,ensure_ascii=False)+'\n')
print('FINAL_OCR_EVIDENCE_WRITTEN',final)


## Full19 — shared compiled inference and provenance

### English
The first validated OCR probe must return `731` + EOS. Reuse its AOT-compiled vision prefill for other image inputs only when PyTree and tensor shapes match. Keep the JAX native layout and sharding validation unchanged. Write a unique incremental per-case JSON sidecar. Additional benchmark repeats remain disabled by default.

### Tiếng Việt
OCR đầu tiên phải trả về `731` và EOS. Chỉ tái sử dụng executable AOT cho ảnh khác khi PyTree và kích thước tensor tương thích. Không bỏ qua kiểm tra sharding/layout. Lưu JSON riêng cho từng trường hợp; các phép lặp benchmark chỉ bật khi có chủ đích.


In [ ]:
# Split-Vision cached runtime reuse and incremental durable provenance.
import json,hashlib,time,traceback,os
from datetime import datetime,timezone
from pathlib import Path
import numpy as np
from PIL import Image as PILImage
assert TRACE['status']=='COMPLETE' and ocr_result['semantic_check'] and ocr_result['eos_reached']
assert ocr_result['generated_text']=='731<|im_end|>'
assert 'compiled' in globals() and 'dynamic_kwargs' in globals()
FULL19_DIR=RUN_ROOT/'cases'
assert not FULL19_DIR.exists(), ('NO_OVERWRITE',str(FULL19_DIR))
FULL19_DIR.mkdir(exist_ok=False)
FULL19_SIDECAR=FULL19_DIR/'full19-case-events.json'
FULL19_FINAL=FULL19_DIR/'full19-final.json'
assert not FULL19_SIDECAR.exists() and not FULL19_FINAL.exists()
stage_manifest=json.loads(STAGE_MANIFEST.read_text())
cache_entries={p.name: p for p in STAGE_CACHE.glob('*') if p.is_file()}
for entry in stage_manifest['entries']:
    assert entry['file'] in cache_entries
generate_cache='jit__generate_jit-8583d9e06f6bcd9b55e0a85553b25aa59f5463d643d81b5fa7b8cd0ad361019b-cache'
assert generate_cache in cache_entries
assert file_sha256(cache_entries[generate_cache])=='ee76270eff0aa04a0128880cd30c44c3b2dfa99cfad69204fae64d4245698c57'
runtime_signature=jax.tree_util.tree_structure(dynamic_kwargs)
EXPECTED_VISION_EMBEDDING_SHAPE=tuple(image_embeddings.shape)
EXPECTED_VISION_PIXEL_SHAPE=tuple(po.pixel_values.shape)
cases_state={'status':'RUNNING','started_utc':datetime.now(timezone.utc).isoformat(),'model':'original BF16 27B',
             'cases':[],'attempted_count':0,'device_count':jax.device_count(),'prefill_length':PREFILL_LEN,
             'max_target_length':MAX_TARGET_LEN,'model_shards':18,'engine_count':1,
             'portable_manifest_sha256':PORTABLE_MANIFEST_SHA256,
             'checkpoint_index_sha256':file_sha256(MODEL_DIR/'model.safetensors.index.json'),
             'maxtext_patch_sha256':PATCH_SHA256,
             'jax_cache_manifest_sha256':file_sha256(STAGE_MANIFEST),
             'run_started_utc':RUN_STARTED_UTC,
             'model_files_attached':str(MODEL_DIR),
             'device_descriptors':[str(x) for x in jax.devices()],
             'ocr731_bootstrap_sha256':file_sha256(ocr_source),'first_ocr_result':ocr_result,
             'actual_rng_seed':None,'rng_seed_note':'Uses MaxEngine RNG and per-call split; no separately attested seed.'}
def save_cases():
    tmp=FULL19_SIDECAR.with_suffix('.writing')
    tmp.write_text(json.dumps(cases_state,indent=2,ensure_ascii=False,default=str)+'\n')
    tmp.replace(FULL19_SIDECAR)
def add_case(group,item):
    assert group in ('text','ocr','rich','photo')
    assert 'id' in item and 'generated_token_ids' in item and 'eos_reached' in item
    assert not any(x['id']==item['id'] for x in cases_state['cases'])
    rec={'group':group,**item}
    cases_state['cases'].append(rec)
    cases_state['active_case']=None
    cases_state['attempted_count']=len(cases_state['cases'])
    save_cases()
    print('FULL19_CASE',len(cases_state['cases']),group,item['id'],
          'EOS',item['eos_reached'],'TOKENS',len(item['generated_token_ids']),flush=True)
# The OCR731 was genuinely generated by the preceding unmodified accepted AOT path.
add_case('ocr',{'id':'ocr-731','image_sha256':file_sha256(ocr_source),
    'prompt':ocr_result['prompt'],'generated_text':ocr_result['generated_text'],
    'generated_token_ids':ocr_result['generated_token_ids'],'eos_reached':ocr_result['eos_reached'],
    'semantic_check':ocr_result['semantic_check'],'vision_seconds':ocr_result['vision_seconds'],
    'prefill_seconds':ocr_result['prefill_seconds'],'decode_step_seconds':ocr_result['decode_step_seconds']})

def begin_case(group,case_id):
    assert not cases_state.get('active_case')
    assert not any(x['id']==case_id for x in cases_state['cases'])
    cases_state['active_case']={'group':group,'id':case_id,'started_utc':datetime.now(timezone.utc).isoformat()}
    save_cases()

def generate_vision_aot_reuse(image_path,prompt,max_decode_steps=224):
    path=Path(image_path);assert path.is_file()
    resized=image_for_inference(path)
    prompt_full=prompt+VISION_SUFFIX
    pd=config._pydantic_config.model_copy(update={'image_path':str(resized),'prompt':prompt_full})
    cfg=pyconfig.HyperParameters(pd)
    prepared=mm_processor.preprocess_mm_data(cfg)
    offset=int(mm_processor.get_image_offsets(config=cfg,processor_output=prepared))
    content=f'<|vision_start|><|image_pad|><|vision_end|>{prompt_full}'
    rendered=tok.apply_chat_template([{'role':'user','content':content}],tokenize=False,add_generation_prompt=True,enable_thinking=False)
    ids=tok(rendered,add_special_tokens=False)['input_ids']
    budget=PREFILL_LEN-offset
    assert len(ids)<=budget,(path.name,len(ids),budget)
    padded=np.asarray(ids+[pad_id]*(budget-len(ids)),dtype=np.int32)
    fused=mm_processor.prepare_text_for_image_fusion(tokens=padded,config=cfg,processor_output=prepared)
    true_length=len(ids)+offset
    assert len(fused)==PREFILL_LEN and MAX_TARGET_LEN-true_length>=max_decode_steps
    attention_mask=(np.arange(len(fused))[None,:]<true_length).astype(np.int32)
    positions,delta=qwen_mm.get_rope_index(input_ids=fused[None,:],image_grid_thw=prepared.pixel_grid_thw,
         attention_mask=attention_mask,spatial_merge_size=cfg.spatial_merge_size_for_vit,
         position_id_per_seconds=cfg.position_id_per_seconds,config=cfg)
    assert tuple(prepared.pixel_values.shape)==EXPECTED_VISION_PIXEL_SHAPE
    start=time.monotonic()
    emb,deep=engine.encode_vision(params=params,images=jnp.asarray(prepared.pixel_values))
    jax.block_until_ready(emb)
    assert tuple(emb.shape)==EXPECTED_VISION_EMBEDDING_SHAPE and not deep
    vision_s=time.monotonic()-start
    engine.rng,key=jax.random.split(engine.rng)
    case_kwargs=dict(dynamic_kwargs)
    case_kwargs.update({'padded_tokens':jnp.asarray(fused,dtype=jnp.int32),
                        'positions':jnp.asarray(positions),'mrope_deltas':jnp.asarray(delta),
                        'image_embeddings':emb,'deepstack_visual_embeds':deep,
                        'rng':key,'true_length':true_length})
    assert jax.tree_util.tree_structure(case_kwargs)==runtime_signature
    for name in ('padded_tokens','positions','mrope_deltas','image_embeddings'):
        old_aval=dynamic_kwargs[name]
        new_aval=case_kwargs[name]
        assert tuple(new_aval.shape)==tuple(old_aval.shape),('AOT_SHAPE_MISMATCH',name,old_aval.shape,new_aval.shape)
        assert new_aval.dtype==old_aval.dtype,('AOT_DTYPE_MISMATCH',name,old_aval.dtype,new_aval.dtype)
    t=time.monotonic()
    prefix,first=compiled(**case_kwargs) # same exact AOT executable validated on OCR731
    jax.block_until_ready(first.data)
    prefill_s=time.monotonic()-t
    t=time.monotonic()
    out,text,steps=_decode(prefix,first,true_length,max_decode_steps)
    decode_s=time.monotonic()-t
    return {'image':str(path),'image_sha256':file_sha256(path),'prompt':prompt_full,
        'generated_text':text,'generated_token_ids':out,'eos_reached':completion_info(out)['eos_reached'],
        'image_offset_tokens':offset,'fused_true_length':true_length,'grid':prepared.pixel_grid_thw.tolist(),
        'pixel_shape':list(prepared.pixel_values.shape),'vision_seconds':vision_s,
        'prefill_seconds':prefill_s,'decode_seconds':decode_s,'decode_step_seconds':steps}
print('FULL19_REUSE_SETUP_READY',len(cases_state['cases']))


## Full19 — second OCR, six Rich Vision, eight Photography

### English
Execute all 15 remaining image cases on the same BF16 engine and Vision AOT executable. Rich Vision/Photography allow up to 448 generated tokens after the prior 224-token cutoff. The source image, prompt, every token, EOS flag and timing are saved case by case. A single unsuccessful case is recorded and stops the notebook, without being counted as acceptance.

### Tiếng Việt
Chạy đủ 15 trường hợp ảnh còn lại bằng cùng engine BF16 và executable AOT. Rich Vision/Photography được phép sinh tối đa 448 token sau khi giới hạn 224 token gây cắt câu trả lời. Mỗi trường hợp ghi ảnh gốc, prompt, token, EOS và thời gian. Một trường hợp thất bại phải được lưu lại rồi dừng; không ghi nhận nghiệm thu giả.


In [ ]:
# Remaining 15 image cases: OCR482 + 6 rich scenes + 8 photographs.
rich_spec=json.loads((EXAMPLES_DIR/'vision_showcase_cases.json').read_text())['cases']
photo_spec=json.loads((EXAMPLES_DIR/'photo_showcase_cases.json').read_text())['cases']
assert len(rich_spec)==6 and len(photo_spec)==8
image_jobs=[('ocr',{'id':'ocr-482','image':'vision_test_card_482_1280x960.png',
                     'prompt':'Read the large number in this image. Answer only the digits.','expected':'482','max_decode_steps':12})]
image_jobs += [('rich',x) for x in rich_spec]
image_jobs += [('photo',x) for x in photo_spec]
assert len(image_jobs)==15
for group,case in image_jobs:
    identifier=case['id']
    path=EXAMPLES_DIR/case['image']
    started=time.monotonic()
    begin_case(group,identifier)
    try:
        row=generate_vision_aot_reuse(path,case['prompt'],max_decode_steps=case.get('max_decode_steps',448))
        row['id']=identifier
        row['group']=group
        if group=='ocr':
            row['expected']=case['expected']
            row['semantic_check']=case['expected'] in row['generated_text']
        if group=='rich':
            hits={x:(x.lower() in row['generated_text'].lower()) for x in case.get('checks',[])}
            row['checks']=hits
            row['check_hits']=sum(hits.values())
            row['check_count']=len(hits)
        row['case_seconds']=round(time.monotonic()-started,3)
        add_case(group,row)
    except BaseException as err:
        cases_state['status']='FAILED_'+identifier
        cases_state['error']={'type':type(err).__name__,'message':str(err),'case_id':identifier}
        save_cases()
        raise
    if group=='ocr' and not row['semantic_check']:
        cases_state['status']='FAILED_OCR482_SEMANTIC';save_cases()
        raise AssertionError(('OCR482_SEMANTIC_FAIL',row['generated_text']))
    if not row['eos_reached']:
        cases_state['status']='FAILED_EOS_'+identifier;save_cases()
        raise AssertionError(('EOS_NOT_REACHED',identifier,len(row['generated_token_ids'])))
print('FULL19_IMAGE_CASES_COMPLETED',len([x for x in cases_state['cases'] if x['group']!='text']))


## Full19 — 3 Text cases and final automated validation

### English
Run the three established text prompts using the same engine, then require 19 distinct results and EOS on all outputs. The rich-image diagnostic counts are reported separately; they are not proof of human-reviewed visual grounding.

### Tiếng Việt
Chạy 3 prompt Text đã định bằng cùng engine rồi kiểm tra đủ 19 kết quả khác nhau và EOS cho tất cả. Chỉ số từ khóa Rich Vision được báo cáo riêng, không thay thế đánh giá trực quan của con người.


In [ ]:
# Three text prompts. Their text-prefill compiled program may differ from vision-prefill.
TEXT_PROMPTS=['2+2','France capital','Ruby language']
for p in TEXT_PROMPTS:
    t=time.monotonic()
    begin_case('text','text-'+p.lower().replace(' ','-').replace('+','plus'))
    try:
        row=generate_text(p,max_decode_steps=96)
        low=row['generated_text'].lower()
        ok=('4' in row['generated_text']) if p=='2+2' else (
            'paris' in low if p=='France capital' else (
                'ruby' in low and ('programming' in low or 'language' in low)))
        row.update({'id':'text-'+p.lower().replace(' ','-').replace('+','plus'),
                    'semantic_check':bool(ok),'case_seconds':round(time.monotonic()-t,3)})
        add_case('text',row)
    except BaseException as exc:
        cases_state['status']='FAILED_TEXT_CASE'
        cases_state['error']={'prompt':p,'type':type(exc).__name__,'message':str(exc)}
        save_cases()
        raise
    if not (row['semantic_check'] and row['eos_reached']):
        cases_state['status']='FAILED_TEXT_SEMANTIC_OR_EOS'
        save_cases()
        raise AssertionError(('TEXT_ACCEPTANCE_FAIL',p,row['generated_text']))
assert len(cases_state['cases'])==19
from collections import Counter
counts=Counter(x['group'] for x in cases_state['cases'])
assert counts=={'text':3,'ocr':2,'rich':6,'photo':8},counts
assert len({x['id'] for x in cases_state['cases']})==19
assert all(x['eos_reached'] for x in cases_state['cases'])
assert all(x.get('semantic_check') for x in cases_state['cases'] if x['group'] in ('text','ocr'))
cases_state['status']='AUTOMATED_19_CASES_EOS_COMPLETED'
cases_state['finished_utc']=datetime.now(timezone.utc).isoformat()
cases_state['total_wall_seconds']=round(time.monotonic()-RUN_WALL_START,3)
cases_state['cold_compile_cache_hit_not_guaranteed']=True
cases_state['group_counts']=dict(counts)
cases_state['all_eos_reached']=True
cases_state['visual_human_review_pending']=True
cases_state['rich_keyword_hits']=sum(x.get('check_hits',0) for x in cases_state['cases'] if x['group']=='rich')
cases_state['rich_keyword_checks']=sum(x.get('check_count',0) for x in cases_state['cases'] if x['group']=='rich')
assert not FULL19_FINAL.exists()
FULL19_FINAL.write_text(json.dumps(cases_state,indent=2,ensure_ascii=False,default=str)+'\n')
save_cases()
print('FULL19_AUTOMATED_ACCEPTANCE',cases_state['status'],'19/19','EOS',True,'VISUAL_REVIEW_PENDING',flush=True)
